In [61]:
import os
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path

def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
    
    return unit_concepts

def analyze_concept_loss(base_dir, model, exp, method, clusters=['Cluster1', 'Cluster2', 'Cluster3'],filename=None):
    """
    Analyze which concepts are lost between different pruning percentages.
    
    Args:
        base_dir: Base directory path
        model: Model name
        exp: Experiment name
        method: Method name (e.g., 'method1', 'method2')
        clusters: List of cluster names
    
    Returns:
        Dictionary containing loss analysis for each cluster
    """
    # Define pruning percentages
    pruning_percentages = ['0.0%Pruned', '25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned']
    
    results = {}
    
    for cluster in clusters:
        print(f"\n{'='*60}")
        print(f"Analyzing {cluster} for {model}/{exp}/{method}")
        print(f"{'='*60}")
        
        cluster_results = {
            'concepts_by_pruning': {},
            'loss_from_baseline': {},
            'sequential_loss': {},
            'cumulative_loss': {}
        }
        
        # Load concepts for each pruning percentage
        baseline_concepts = None
        prev_concepts = None
        
        for pruning_pct in pruning_percentages:
            filepath = os.path.join(base_dir, model, exp, method,filename, 'Expls', pruning_pct, f'{cluster}IOUS1024N.csv')
            
            if not os.path.exists(filepath):
                print(f"Warning: File not found - {filepath}")
                continue
            
            # Load unit-concept mappings
            unit_concepts = load_csv_data(filepath)
            
            # Get all unique concepts at this pruning level
            all_concepts = set()
            for concepts in unit_concepts.values():
                all_concepts.update(concepts)
            
            cluster_results['concepts_by_pruning'][pruning_pct] = {
                'all_concepts': all_concepts,
                'num_concepts': len(all_concepts),
                'unit_concepts': unit_concepts
            }
            
            # Set baseline (0.0% pruned)
            if baseline_concepts is None:
                baseline_concepts = all_concepts
                print(f"\n{pruning_pct}: {len(all_concepts)} concepts (BASELINE)")
            else:
                # Calculate loss from baseline
                lost_from_baseline = baseline_concepts - all_concepts
                retained_from_baseline = baseline_concepts & all_concepts
                
                cluster_results['loss_from_baseline'][pruning_pct] = {
                    'lost_concepts': lost_from_baseline,
                    'num_lost': len(lost_from_baseline),
                    'retained_concepts': retained_from_baseline,
                    'num_retained': len(retained_from_baseline),
                    'loss_percentage': (len(lost_from_baseline) / len(baseline_concepts) * 100) if baseline_concepts else 0
                }
                
                print(f"\n{pruning_pct}: {len(all_concepts)} concepts")
                print(f"  Lost from baseline: {len(lost_from_baseline)} ({cluster_results['loss_from_baseline'][pruning_pct]['loss_percentage']:.2f}%)")
                print(f"  Retained from baseline: {len(retained_from_baseline)}")
                
                # Calculate sequential loss (compared to previous pruning level)
                if prev_concepts is not None:
                    lost_sequential = prev_concepts - all_concepts
                    
                    cluster_results['sequential_loss'][pruning_pct] = {
                        'lost_concepts': lost_sequential,
                        'num_lost': len(lost_sequential),
                        'loss_percentage': (len(lost_sequential) / len(prev_concepts) * 100) if prev_concepts else 0
                    }
                    
                    print(f"  Lost since previous: {len(lost_sequential)} ({cluster_results['sequential_loss'][pruning_pct]['loss_percentage']:.2f}%)")
            
            prev_concepts = all_concepts
        
        results[cluster] = cluster_results
    
    return results
from collections import defaultdict
def generate_loss_summary(loss_results, save_path=None):
    """
    Generate a summary table of concept loss across pruning percentages.
    
    Args:
        loss_results: Results from analyze_concept_loss
        save_path: Optional path to save CSV summary
    """
    summary_data = []
    raw_concepts = defaultdict(lambda: defaultdict(list))
    max_s=defaultdict(int)
    for cluster, cluster_data in loss_results.items():
        for pct in cluster_data['loss_from_baseline'].keys():
            raw_concepts[cluster][pct]= list(cluster_data['loss_from_baseline'][pct]['lost_concepts'])
            max_s[cluster] = max(max_s[cluster], len(raw_concepts[cluster][pct]))
        for pruning_pct in cluster_data['concepts_by_pruning'].keys():
            row = {
                'Cluster': cluster,
                'Pruning_Percentage': pruning_pct,
                'Total_Concepts': cluster_data['concepts_by_pruning'][pruning_pct]['num_concepts']
            }
            
            if pruning_pct in cluster_data['loss_from_baseline']:
                row['Lost_from_Baseline'] = cluster_data['loss_from_baseline'][pruning_pct]['num_lost']
                row['Lost_from_Baseline_Pct'] = cluster_data['loss_from_baseline'][pruning_pct]['loss_percentage']
                row['Retained_from_Baseline'] = cluster_data['loss_from_baseline'][pruning_pct]['num_retained']
            else:
                row['Lost_from_Baseline'] = 0
                row['Lost_from_Baseline_Pct'] = 0
                row['Retained_from_Baseline'] = row['Total_Concepts']
            
            if pruning_pct in cluster_data['sequential_loss']:
                row['Lost_Sequential'] = cluster_data['sequential_loss'][pruning_pct]['num_lost']
                row['Lost_Sequential_Pct'] = cluster_data['sequential_loss'][pruning_pct]['loss_percentage']
            else:
                row['Lost_Sequential'] = 0
                row['Lost_Sequential_Pct'] = 0
            
            summary_data.append(row)
    
    for cluster in raw_concepts:
        for pct in raw_concepts[cluster]:
            dif = max_s[cluster] - len(raw_concepts[cluster][pct]) 
            if dif > 0:
                for i in range(dif):
                    raw_concepts[cluster][pct].append('')
                
        pd.DataFrame({k: sorted(v) for k, v in raw_concepts[cluster].items()}).to_csv(f"{save_path}_Cluster{cluster}_concepts_lost.csv")
    summary_df = pd.DataFrame(summary_data)
    pd.DataFrame(raw_concepts).to_csv("Raw_concepts_lost_to_pruning.csv")
    if save_path:
        summary_df.to_csv(save_path, index=False)
        print(f"\nSummary saved to {save_path}_concept_loss_summary.csv")
    
    return summary_df

def get_lost_concepts_details(loss_results, cluster, pruning_pct):
    """
    Get detailed list of lost concepts for a specific cluster and pruning percentage.
    
    Args:
        loss_results: Results from analyze_concept_loss
        cluster: Cluster name
        pruning_pct: Pruning percentage (e.g., '25.0%Pruned')
    
    Returns:
        Set of lost concepts
    """
    if cluster not in loss_results:
        print(f"Cluster {cluster} not found in results")
        return set()
    
    if pruning_pct not in loss_results[cluster]['loss_from_baseline']:
        print(f"Pruning percentage {pruning_pct} not found for {cluster}")
        return set()
    
    return loss_results[cluster]['loss_from_baseline'][pruning_pct]['lost_concepts']

# Example usage:
if __name__ == "__main__":
    # Set your paths
    base_dir = "/workspace/CCE_NLI"
    model = "LLAMA"
    exp = "exp"
    method = "lottery_ticket"
    filename='Run0.25_2'
    
    # Analyze concept loss
    loss_results = analyze_concept_loss(base_dir, model, exp, method, filename=filename)
    
    # Generate summary table
    summary_df = generate_loss_summary(loss_results, save_path=f'{model}_{method}_{filename}')
    print("\nSummary Table:")
    print(summary_df)

    # Get specific lost concepts
    lost_concepts_25 = get_lost_concepts_details(loss_results, 'Cluster1', '25.0%Pruned')
    print(f"\nConcepts lost at 25% pruning in cluster1: {len(lost_concepts_25)}")
    print(f"Examples: {list(lost_concepts_25)[:10]}")


Analyzing Cluster1 for LLAMA/exp/lottery_ticket

0.0%Pruned: 194 concepts (BASELINE)

25.0%Pruned: 135 concepts
  Lost from baseline: 80 (41.24%)
  Retained from baseline: 114
  Lost since previous: 80 (41.24%)

43.75%Pruned: 129 concepts
  Lost from baseline: 84 (43.30%)
  Retained from baseline: 110
  Lost since previous: 35 (25.93%)

57.812%Pruned: 158 concepts
  Lost from baseline: 77 (39.69%)
  Retained from baseline: 117
  Lost since previous: 28 (21.71%)

68.359%Pruned: 180 concepts
  Lost from baseline: 57 (29.38%)
  Retained from baseline: 137
  Lost since previous: 42 (26.58%)

76.27%Pruned: 170 concepts
  Lost from baseline: 65 (33.51%)
  Retained from baseline: 129
  Lost since previous: 56 (31.11%)

Analyzing Cluster2 for LLAMA/exp/lottery_ticket

0.0%Pruned: 233 concepts (BASELINE)

25.0%Pruned: 206 concepts
  Lost from baseline: 79 (33.91%)
  Retained from baseline: 154
  Lost since previous: 79 (33.91%)

43.75%Pruned: 221 concepts
  Lost from baseline: 76 (32.62%)
  Re

In [71]:
c1_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster1_concepts_lost.csv")
c1_lost_lth = pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster1_concepts_lost.csv")
lost=set(c1_lost_lth['25.0%Pruned'])
for i in c1_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c1_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c1_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c1_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 32
Num lost in wanda and lth: 13
{'hyp:tok:green', 'pre:tok:top', 'pre:tok:there', 'pre:tok:lake', 'pre:tok:players', 'pre:tok:tree', 'hyp:tok:around', 'pre:tok:wall', 'pre:tok:performing', 'hyp:tok:driving', 'hyp:tok:female', 'hyp:tok:cats', 'pre:tok:out'}


In [67]:
len(preserved) #% of concepts that are lost once you rpune (lost at all iters)

33

In [72]:
c2_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster2_concepts_lost.csv")
c2_lost_lth= pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster2_concepts_lost.csv")
lost=set(c2_lost_lth['25.0%Pruned'])
for i in c2_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c2_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c2_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c2_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 21
Num lost in wanda and lth: 8
{'pre:tok:church', 'pre:tok:vendor', 'pre:tok:workers', 'hyp:tok:air', 'hyp:tok:guitar', 'pre:tok:jacket', 'hyp:tok:couple', 'pre:tag:ex'}


In [73]:
c3_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster3_concepts_lost.csv")
c3_lost_lth = pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster3_concepts_lost.csv")
lost=set(c3_lost_lth['25.0%Pruned'])
for i in c3_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c3_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c3_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c3_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 27
Num lost in wanda and lth: 10
{'hyp:tok:construction', 'pre:tok:graffiti', 'hyp:tok:summer', 'pre:tok:skier', 'pre:tok:board', 'pre:tok:surfing', 'pre:tok:shoulders', 'pre:tok:martial', 'pre:tok:flying', 'hyp:tok:runs'}
